# 10.11 圖找文和文找圖，為什麼要分開算？


同一張配對分數表，找文字時在每橫列選候選；找圖片時則在每直欄選候選。先看手設表 `[[2.0,0.1],[0.4,1.8]]`：橫列代表圖，直欄代表文字，正確配對都是索引 0、1。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch.nn import functional as F

scores = torch.tensor([[2.0, 0.1], [0.4, 1.8]], requires_grad=True)
labels = torch.arange(2)
image_to_text = F.cross_entropy(scores, labels)
text_to_image = F.cross_entropy(scores.T, labels)
loss = (image_to_text + text_to_image) / 2
loss.backward()
print("圖找文", round(image_to_text.item(), 4))
print("文找圖", round(text_to_image.item(), 4))
print("平均", round(loss.item(), 4))
print("梯度符號", torch.sign(scores.grad).tolist())

原表算「圖找文」，轉置 `scores.T` 讓文字變橫列、圖變直欄，算「文找圖」。兩個交叉熵約 0.1799、0.1758，平均約 0.1779；梯度符號是 `[[-1,1],[1,-1]]`。同一分數因此收到兩方向候選競爭的影響。

兩項不必相等：圖找文比較 2.0 對 0.1、0.4 對 1.8；文找圖比較 2.0 對 0.4、0.1 對 1.8，差距不同。若只用對稱表，很容易把雙向誤讀為同一算式重複兩次。

雙向平均表達兩種查詢都重要，但不保證每次效果勝過單向。原六類小實驗中，兩種版本的最後題、兩個方向都為 6/6，沒有觀察到答對數優勢；驗證也非全部正確。這只能支持該有限候選配對，不能當大型檢索比較。

檢索是在既有卡片中選一張，生成描述要逐步寫出文字。兩者可共享特徵，能力證據仍需分開。相同描述對多張圖時，也要沿用上一節的多正確配對規則。

練習改成 `(2*image_to_text + text_to_image)/3`，預期形狀不變、代價更靠近圖找文。它改的是重視程度，沒有增加新圖片資訊。

<details>
<summary>回顧與查證</summary>

可回顧：[10.10把配對當分類](https://birdhackor.github.io/tiny-perceptron-vlm/10.10.html)、[W.5矩陣轉置](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/contrastive.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
